In [1]:
!pip install seaborn matplotlib scikit-learn


In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150


# 1. LOAD MODEL RESULTS

print("Loading model results...")

with open('C:\\Users\\My PC\\OneDrive\\Documents\\AI Project\\used-car-price-predictor\\models\\xgboost_results.json', 'r') as f:
    xgb_results = json.load(f)

with open('C:\\Users\\My PC\\OneDrive\\Documents\\AI Project\\used-car-price-predictor\\models\\lightgbm_results.json', 'r') as f:
    lgb_results = json.load(f)

print("Results loaded successfully!\n")

# Create evaluation directory
eval_dir = Path('models/evaluation')
eval_dir.mkdir(parents=True, exist_ok=True)


# 2. MODEL COMPARISON TABLE

print("="*80)
print("MODEL COMPARISON - Performance Metrics")
print("="*80)

comparison_data = {
    'Model': ['XGBoost', 'LightGBM'],
    'Train MAE': [
        f"Rs {xgb_results['train_metrics']['mae']:,.0f}",
        f"Rs {lgb_results['train_metrics']['mae']:,.0f}"
    ],
    'Train R²': [
        f"{xgb_results['train_metrics']['r2']:.4f}",
        f"{lgb_results['train_metrics']['r2']:.4f}"
    ],
    'Val MAE': [
        f"Rs {xgb_results['val_metrics']['mae']:,.0f}",
        f"Rs {lgb_results['val_metrics']['mae']:,.0f}"
    ],
    'Val R²': [
        f"{xgb_results['val_metrics']['r2']:.4f}",
        f"{lgb_results['val_metrics']['r2']:.4f}"
    ],
    'Test MAE': [
        f"Rs {xgb_results['test_metrics']['mae']:,.0f}",
        f"Rs {lgb_results['test_metrics']['mae']:,.0f}"
    ],
    'Test RMSE': [
        f"Rs {xgb_results['test_metrics']['rmse']:,.0f}",
        f"Rs {lgb_results['test_metrics']['rmse']:,.0f}"
    ],
    'Test R²': [
        f"{xgb_results['test_metrics']['r2']:.4f}",
        f"{lgb_results['test_metrics']['r2']:.4f}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("="*80 + "\n")

# Determine best model
xgb_r2 = xgb_results['test_metrics']['r2']
lgb_r2 = lgb_results['test_metrics']['r2']
best_model = 'LightGBM' if lgb_r2 > xgb_r2 else 'XGBoost'
improvement = abs(lgb_r2 - xgb_r2) * 100

print(f" Best Model: {best_model}")
print(f"   Performance difference: {improvement:.2f}% in R² score\n")


# 3. VISUALIZATION 1: FEATURE IMPORTANCE COMPARISON

print("Creating feature importance comparison plot...")

xgb_fi = pd.DataFrame(xgb_results['feature_importance']).head(10)
lgb_fi = pd.DataFrame(lgb_results['feature_importance']).head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# XGBoost feature importance
colors_xgb = plt.cm.Blues(np.linspace(0.4, 0.8, len(xgb_fi)))
ax1.barh(xgb_fi['feature'][::-1], xgb_fi['importance'][::-1], 
         color=colors_xgb[::-1], edgecolor='black', linewidth=0.5)
ax1.set_xlabel('Importance Score', fontsize=12, fontweight='bold')
ax1.set_title('XGBoost - Top 10 Features', fontsize=14, fontweight='bold', pad=15)
ax1.grid(axis='x', alpha=0.3, linestyle='--')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# LightGBM feature importance
colors_lgb = plt.cm.Oranges(np.linspace(0.4, 0.8, len(lgb_fi)))
ax2.barh(lgb_fi['feature'][::-1], lgb_fi['importance'][::-1], 
         color=colors_lgb[::-1], edgecolor='black', linewidth=0.5)
ax2.set_xlabel('Importance Score', fontsize=12, fontweight='bold')
ax2.set_title('LightGBM - Top 10 Features', fontsize=14, fontweight='bold', pad=15)
ax2.grid(axis='x', alpha=0.3, linestyle='--')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(eval_dir / 'feature_importance_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {eval_dir / 'feature_importance_comparison.png'}\n")


# 4. VISUALIZATION 2: R² SCORE COMPARISON

print("Creating R² score comparison plot...")

fig, ax = plt.subplots(figsize=(10, 6))

datasets = ['Train', 'Validation', 'Test']
xgb_scores = [
    xgb_results['train_metrics']['r2'],
    xgb_results['val_metrics']['r2'],
    xgb_results['test_metrics']['r2']
]
lgb_scores = [
    lgb_results['train_metrics']['r2'],
    lgb_results['val_metrics']['r2'],
    lgb_results['test_metrics']['r2']
]

x = np.arange(len(datasets))
width = 0.35

bars1 = ax.bar(x - width/2, xgb_scores, width, label='XGBoost', 
               color='steelblue', edgecolor='black', linewidth=1)
bars2 = ax.bar(x + width/2, lgb_scores, width, label='LightGBM', 
               color='coral', edgecolor='black', linewidth=1)

ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison (R² Score)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(datasets, fontsize=11)
ax.legend(fontsize=11, frameon=True, shadow=True)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim([0.85, 1.0])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(eval_dir / 'r2_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {eval_dir / 'r2_comparison.png'}\n")

# 5. VISUALIZATION 3: MAE & RMSE COMPARISON

print("Creating MAE and RMSE comparison plots...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MAE Comparison
xgb_mae = [
    xgb_results['train_metrics']['mae'],
    xgb_results['val_metrics']['mae'],
    xgb_results['test_metrics']['mae']
]
lgb_mae = [
    lgb_results['train_metrics']['mae'],
    lgb_results['val_metrics']['mae'],
    lgb_results['test_metrics']['mae']
]

bars1 = ax1.bar(x - width/2, xgb_mae, width, label='XGBoost', 
                color='steelblue', edgecolor='black', linewidth=1)
bars2 = ax1.bar(x + width/2, lgb_mae, width, label='LightGBM', 
                color='coral', edgecolor='black', linewidth=1)

ax1.set_ylabel('MAE (Rs)', fontsize=11, fontweight='bold')
ax1.set_title('Mean Absolute Error Comparison', fontsize=12, fontweight='bold', pad=10)
ax1.set_xticks(x)
ax1.set_xticklabels(datasets)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Add value labels for MAE
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height/1000:.0f}K', ha='center', va='bottom', fontsize=8)

# RMSE Comparison
xgb_rmse = [
    xgb_results['train_metrics']['rmse'],
    xgb_results['val_metrics']['rmse'],
    xgb_results['test_metrics']['rmse']
]
lgb_rmse = [
    lgb_results['train_metrics']['rmse'],
    lgb_results['val_metrics']['rmse'],
    lgb_results['test_metrics']['rmse']
]

bars3 = ax2.bar(x - width/2, xgb_rmse, width, label='XGBoost', 
                color='steelblue', edgecolor='black', linewidth=1)
bars4 = ax2.bar(x + width/2, lgb_rmse, width, label='LightGBM', 
                color='coral', edgecolor='black', linewidth=1)

ax2.set_ylabel('RMSE (Rs)', fontsize=11, fontweight='bold')
ax2.set_title('Root Mean Squared Error Comparison', fontsize=12, fontweight='bold', pad=10)
ax2.set_xticks(x)
ax2.set_xticklabels(datasets)
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Add value labels for RMSE
for bars in [bars3, bars4]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height/1000:.0f}K', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(eval_dir / 'mae_rmse_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print(f" Saved: {eval_dir / 'mae_rmse_comparison.png'}\n")


# 6. HYPERPARAMETER COMPARISON TABLE

print("="*80)
print("HYPERPARAMETER COMPARISON")
print("="*80)

print("\nXGBoost Best Parameters:")
for param, value in xgb_results['best_params'].items():
    print(f"  • {param:20s}: {value}")

print("\nLightGBM Best Parameters:")
for param, value in lgb_results['best_params'].items():
    print(f"  • {param:20s}: {value}")

print("="*80 + "\n")


# 7. COMPREHENSIVE SUMMARY REPORT

summary = f"""
{'='*80}
MODEL EVALUATION SUMMARY - PakWheels Car Price Predictor
{'='*80}

BEST MODEL: {best_model} ({'+'if lgb_r2 > xgb_r2 else '-'}{improvement:.2f}% vs {'XGBoost' if best_model == 'LightGBM' else 'LightGBM'})

{'='*80}
XGBOOST PERFORMANCE
{'='*80}

Training Set:
  • MAE:  Rs {xgb_results['train_metrics']['mae']:,.0f}
  • RMSE: Rs {xgb_results['train_metrics']['rmse']:,.0f}
  • R²:   {xgb_results['train_metrics']['r2']:.4f}

Validation Set:
  • MAE:  Rs {xgb_results['val_metrics']['mae']:,.0f}
  • RMSE: Rs {xgb_results['val_metrics']['rmse']:,.0f}
  • R²:   {xgb_results['val_metrics']['r2']:.4f}

Test Set:
  • MAE:  Rs {xgb_results['test_metrics']['mae']:,.0f}
  • RMSE: Rs {xgb_results['test_metrics']['rmse']:,.0f}
  • R²:   {xgb_results['test_metrics']['r2']:.4f}

Best Hyperparameters:
  • max_depth: {xgb_results['best_params']['max_depth']}
  • learning_rate: {xgb_results['best_params']['learning_rate']}
  • n_estimators: {xgb_results['best_params']['n_estimators']}
  • min_child_weight: {xgb_results['best_params']['min_child_weight']}
  • subsample: {xgb_results['best_params']['subsample']}
  • colsample_bytree: {xgb_results['best_params']['colsample_bytree']}

Top 3 Important Features:
  1. {xgb_fi.iloc[0]['feature']}
  2. {xgb_fi.iloc[1]['feature']}
  3. {xgb_fi.iloc[2]['feature']}

{'='*80}
LIGHTGBM PERFORMANCE
{'='*80}

Training Set:
  • MAE:  Rs {lgb_results['train_metrics']['mae']:,.0f}
  • RMSE: Rs {lgb_results['train_metrics']['rmse']:,.0f}
  • R²:   {lgb_results['train_metrics']['r2']:.4f}

Validation Set:
  • MAE:  Rs {lgb_results['val_metrics']['mae']:,.0f}
  • RMSE: Rs {lgb_results['val_metrics']['rmse']:,.0f}
  • R²:   {lgb_results['val_metrics']['r2']:.4f}

Test Set:
  • MAE:  Rs {lgb_results['test_metrics']['mae']:,.0f}
  • RMSE: Rs {lgb_results['test_metrics']['rmse']:,.0f}
  • R²:   {lgb_results['test_metrics']['r2']:.4f}

Best Hyperparameters:
  • max_depth: {lgb_results['best_params']['max_depth']}
  • learning_rate: {lgb_results['best_params']['learning_rate']}
  • n_estimators: {lgb_results['best_params']['n_estimators']}
  • num_leaves: {lgb_results['best_params']['num_leaves']}
  • min_child_samples: {lgb_results['best_params']['min_child_samples']}
  • subsample: {lgb_results['best_params']['subsample']}

Top 3 Important Features:
  1. {lgb_fi.iloc[0]['feature']}
  2. {lgb_fi.iloc[1]['feature']}
  3. {lgb_fi.iloc[2]['feature']}

{'='*80}
KEY FINDINGS
{'='*80}

Model Selection:
  → {best_model} performs better with R² = {max(xgb_r2, lgb_r2):.4f}
  → Lower MAE indicates better average prediction accuracy
  → Both models show good generalization (Train R² ~ Test R²)

Feature Importance:
  → XGBoost prioritizes: {xgb_fi.iloc[0]['feature']}, {xgb_fi.iloc[1]['feature']}
  → LightGBM prioritizes: {lgb_fi.iloc[0]['feature']}, {lgb_fi.iloc[1]['feature']}
  → Both models identify engine and brand features as critical

Performance Analysis:
  → Average prediction error: ~Rs {lgb_results['test_metrics']['mae']:,.0f}
  → 87% of price variance explained (R² = 0.87)
  → Suitable for deployment in production environment

{'='*80}
FILES GENERATED
{'='*80}

Visualizations:
   feature_importance_comparison.png
   r2_comparison.png
   mae_rmse_comparison.png

Models:
   xgboost_model.pkl
   lightgbm_model.pkl
   label_encoders.pkl

Results:
   xgboost_results.json
   lightgbm_results.json
   evaluation_summary.txt

{'='*80}
End of Evaluation Report
{'='*80}
"""

# Save summary report
with open(eval_dir / 'evaluation_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary)

print(summary)
print(f"\n Summary saved: {eval_dir / 'evaluation_summary.txt'}")
print(f"\n Model evaluation complete! All files saved to: {eval_dir}")

Loading model results...
Results loaded successfully!

MODEL COMPARISON - Performance Metrics
   Model  Train MAE Train R²    Val MAE Val R²   Test MAE    Test RMSE Test R²
 XGBoost Rs 256,881   0.9908 Rs 515,832 0.9072 Rs 571,006 Rs 1,844,519  0.8614
LightGBM Rs 374,429   0.9681 Rs 538,849 0.9030 Rs 562,417 Rs 1,764,932  0.8731

 Best Model: LightGBM
   Performance difference: 1.17% in R² score

Creating feature importance comparison plot...
Saved: models\evaluation\feature_importance_comparison.png

Creating R² score comparison plot...
Saved: models\evaluation\r2_comparison.png

Creating MAE and RMSE comparison plots...
 Saved: models\evaluation\mae_rmse_comparison.png

HYPERPARAMETER COMPARISON

XGBoost Best Parameters:
  • colsample_bytree    : 0.8
  • learning_rate       : 0.1
  • max_depth           : 7
  • min_child_weight    : 1
  • n_estimators        : 300
  • subsample           : 1.0

LightGBM Best Parameters:
  • learning_rate       : 0.05
  • max_depth           : -1
  • 